# TalentDesk, Module 3 Section 1 Lab (Exercise): Project Memory and Path-Scoped Rules

A hands-on exercise on how Claude Code assembles project memory. It combines the two M3S1 skills:
the **CLAUDE.md hierarchy** (user, project, directory) with **@import** and the `/memory` view (Lab
1), and **path-scoped `.claude/rules/`** that load only when Claude touches a matching file, saving
context (Lab 2). You fill in four short `TODO` blocks over a config **sandbox** that uses a fake home,
so your real `~/.claude` is never touched. Everything is testable offline, and a live **Claude Agent
SDK** cell loads the project memory. Runs **Sonnet** (`claude-sonnet-4-6`).

## The real-world scenario

TalentDesk's teammates cannot see your personal `~/.claude/CLAUDE.md`, so team conventions (how to
word a candidate message, the screening naming style, scheduling etiquette) have to live in the repo.
A single giant `CLAUDE.md` gets unreadable, so you split shared standards into files and pull them in
with `@import`, and you keep screening-specific and scheduling-specific rules near the code they
govern. When behaviour looks wrong, `/memory` tells you what actually loaded.

Then there is cost. Putting every convention in one always-on file means every session pays for every
rule, even when editing an unrelated file. **Path-scoped rules** fix this: the screening rule loads
only for screening files, the scheduling rule only for scheduling files, and a shared style rule loads
everywhere.

The question this lab answers: **how do the user, project, and directory files combine, how does
`@import` keep memory modular, and how do path globs decide which rules load and how much context that
saves?**

## Objectives

- Build a **user + project** `CLAUDE.md`, resolve **@import**, and see how a more specific scope
  overrides a broader one.
- Inspect what loaded the way **/memory** would.
- Write `.claude/rules/` with a **paths** field, validate that each file loads exactly its rules, and
  compare the token cost against a **monolithic** file.

## The outcome you should reach

By the end you will have:

- an `@import` resolver that inlines shared standards, and a conflict resolver where the project's
  indentation rule overrides the user's;
- a `/memory`-style listing of the active files for a target;
- a glob matcher and a `rules_for(file)` resolver where a screening file loads the screening rule, a
  test file loads the test rule, and a plain file loads only the shared style rule;
- and a token comparison showing path scoping loads less than a monolithic file.

Target time: **20 to 30 minutes.** Four small `TODO` blocks, all testable offline. The live cell needs
a real key and Node.js 18+.

## How to run

Run top to bottom. Building the sandbox and the pure-Python cells run anywhere and use a fake home, so
your real `~/.claude` is untouched. The live cell loads project memory through Claude, so paste a real
key into **Setup 2/3** and re-run from the top; **Node.js 18+** is needed for the Agent SDK.

## 0. Setup

**This cell:** installs the packages. `pyyaml` parses the rule frontmatter; the Agent SDK drives
the live project-memory cell and needs Node.js 18+.

In [ ]:
# ===== SETUP 1/3 - install the packages =====
%pip install -q claude-agent-sdk anthropic python-dotenv pyyaml

**This cell:** imports, the model, the `RUN_LIVE` switch, and `run_async()` for the live
cell.

In [ ]:
# ===== SETUP 2/3 - imports, the model, the switch, and an async runner =====
import os                                       # filesystem paths for the sandbox
import re                                       # detect @import lines and match globs
import sys                                       # detect Windows (special event loop)
import yaml                                      # parse the rule frontmatter
import textwrap                                  # keeps the embedded file bodies readable
import asyncio                                  # the Agent SDK is async; we drive it ourselves
import threading                                # run that async loop in a side thread (notebook-safe)

try:                                            # load a .env file if present
    from dotenv import load_dotenv             #   import the loader
    load_dotenv()                               #   read .env into environment variables
except Exception:                               # not installed? that is fine
    pass                                        #   set the key another way

MODEL = "claude-sonnet-4-6"                      # the Sonnet model the live cell will use

os.environ.setdefault("ANTHROPIC_API_KEY", "sk-ant-...")     # placeholder unless you set a real key
_key = os.environ["ANTHROPIC_API_KEY"]           # read whatever key is set
RUN_LIVE = _key.startswith("sk-ant-") and _key != "sk-ant-..."   # True only for a real key

def run_async(make_coro):                        # run any async Agent SDK call, notebook-safe
    box = {}
    def worker():
        loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:    box["value"] = loop.run_until_complete(make_coro())
        except Exception as e: box["error"] = e
        finally: loop.close()
    t = threading.Thread(target=worker); t.start(); t.join()
    if "error" in box: raise box["error"]
    return box.get("value")

print("live model calls:", "ON" if RUN_LIVE else "OFF (sandbox cells run offline)")

**This cell:** builds a **sandbox** with a fake home and a project (provided). It writes a
user-level `CLAUDE.md`, a project `CLAUDE.md` that `@import`s two standards files, a directory-level
`CLAUDE.md` for the screening folder, and three path-scoped rules. Using a fake home means nothing here
touches your real configuration.

In [ ]:
# ===== SETUP 3/3 - build the sandbox hierarchy (fake home + project) =====
SANDBOX = os.path.join(os.getcwd(), "talentdesk_config_sandbox")
HOME = os.path.join(SANDBOX, "home")               # a FAKE home, not your real ~
PROJECT = os.path.join(SANDBOX, "project")         # the sample project root

def write(path, content):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write(content)

# --- the CLAUDE.md hierarchy ---
write(os.path.join(HOME, ".claude/CLAUDE.md"), textwrap.dedent("""\
    # User memory (personal, not shared)
    - Use 2-space indentation.
    - Keep replies brief.
    """))
write(os.path.join(PROJECT, ".claude/CLAUDE.md"), textwrap.dedent("""\
    # TalentDesk project memory
    - Use 4-space indentation.
    @standards/tone.md
    @standards/naming.md
    """))
write(os.path.join(PROJECT, ".claude/standards/tone.md"), "# Tone\n- Candidate messages are warm and specific.\n")
write(os.path.join(PROJECT, ".claude/standards/naming.md"), "# Naming\n- Name screening helpers with a screen_ prefix.\n")
write(os.path.join(PROJECT, "talentdesk/screening/CLAUDE.md"), "# Screening folder memory\n- Every screen logs its reason.\n")

# --- path-scoped rules ---
write(os.path.join(PROJECT, ".claude/rules/screening.md"),
      '---\npaths:\n  - "talentdesk/screening/**/*.py"\n---\n'
      '# Screening rules\n- Ground every decision in the candidate facts.\n')
write(os.path.join(PROJECT, ".claude/rules/tests.md"),
      '---\npaths:\n  - "**/test_*.py"\n---\n'
      '# Testing rules\n- Name tests test_*.py; mock the ATS, never call it.\n')
write(os.path.join(PROJECT, ".claude/rules/style.md"),
      '# Style rules (no paths -> always loaded)\n- Prefer clear names over comments.\n')

# --- a few source files to match against ---
for f in ["talentdesk/screening/bar.py", "talentdesk/scheduling/slots.py",
          "talentdesk/util/text.py", "tests/test_screening.py"]:
    write(os.path.join(PROJECT, f), "# " + f + "\n")
print("sandbox at", SANDBOX)

### The load order

CLAUDE.md files combine from **broadest to most specific**, so a later layer overrides an earlier one:

- **User** `~/.claude/CLAUDE.md`: personal, loaded first, **not shared** with your team.
- **Project** `.claude/CLAUDE.md`: shared via Git, loaded after the user file.
- **Directory** `CLAUDE.md` in a subfolder: loaded **on demand** when Claude works in that subtree.

Because the user file is not shared, team conventions belong in the project file and in
`.claude/rules/`.

---

### 🎯 Part A - see how memory is assembled

**TODO 1 (about 5 minutes).** Complete `resolve_imports()`, the `@import` resolver. A line that is
just `@path` pulls in that file's contents, relative to the importing file's folder; other lines pass
through unchanged. Imports may nest (the depth guard is provided).

In [ ]:
# ===== TODO 1 - resolve @import directives =====
IMPORT_RE = re.compile(r"^\s*@(\S+)\s*$")          # a line that is only "@some/path"

def resolve_imports(path, depth=0):                # inline @imports, relative to each file's folder
    if depth > 4:
        return "[import depth exceeded]"
    base = os.path.dirname(path)                     # imports are relative to the importing file
    lines = []
    for line in open(path).read().splitlines():
        m = IMPORT_RE.match(line)
        if m:
            # 👉 TODO 1a: it is an @import -> inline the target file, recursing:
            #    lines.append(resolve_imports(os.path.join(base, m.group(1)), depth + 1))
            pass
        else:
            # 👉 TODO 1b: an ordinary line -> keep it as-is:
            #    lines.append(line)
            pass
    return "\n".join(lines)

print(resolve_imports(os.path.join(PROJECT, ".claude/CLAUDE.md")))

**Self-check (offline).** The inlined project memory should contain the tone and naming standards
that were imported.

In [ ]:
# ===== self-check for TODO 1 =====
resolved = resolve_imports(os.path.join(PROJECT, ".claude/CLAUDE.md"))
assert "warm and specific" in resolved, "tone.md should be inlined"
assert "screen_ prefix" in resolved, "naming.md should be inlined"
assert "@standards" not in resolved, "the @import lines should be replaced, not kept"
print("TODO 1 checks passed")

**This cell:** note what `@import` does and does not do (provided). It **organizes** your
instructions into separate files, but the imported files still load at launch, so it does **not** reduce
context cost. The mechanism that saves context is path-scoped `.claude/rules/`, in Part B.

In [ ]:
# ===== imports organize, but still load at launch (provided) =====
imported = [".claude/standards/tone.md", ".claude/standards/naming.md"]
print("imported files still loaded at launch:", imported)
print("project memory size after inlining (chars):", len(resolve_imports(os.path.join(PROJECT, ".claude/CLAUDE.md"))))

**This cell:** the **effective config** for a target file (provided): user, then project, then the
directory file when the target is under `talentdesk/screening/`. The later layer wins on conflicts.

In [ ]:
# ===== assemble the layers for a target file (provided) =====
def effective_config(target):                      # target path -> ordered (scope, source) layers
    layers = [("user", os.path.join(HOME, ".claude/CLAUDE.md")),
              ("project", os.path.join(PROJECT, ".claude/CLAUDE.md"))]
    if target.startswith("talentdesk/screening/"):  # directory memory loads on demand for this subtree
        layers.append(("dir:screening", os.path.join(PROJECT, "talentdesk/screening/CLAUDE.md")))
    return layers

for target in ["talentdesk/screening/bar.py", "talentdesk/util/text.py"]:
    print(f"  {target:28} -> {[s for s, _ in effective_config(target)]}")

**TODO 2 (about 5 minutes).** Complete `indent_rule()`, the conflict resolver. Walk the layers
broadest to most specific and let the **last** matching value win. For the project layer, read through
`resolve_imports` (so inlined standards are visible); for other layers, read the file directly. The
project's 4-space rule should override the user's 2-space rule.

In [ ]:
# ===== TODO 2 - more specific overrides broader =====
def indent_rule(target):                           # walk layers; the last matching value wins
    winner = None
    for scope, src in effective_config(target):
        text = resolve_imports(src) if scope == "project" else open(src).read()
        for line in text.splitlines():
            m = re.search(r"Use (\d+)-space indentation", line)
            if m:
                # 👉 TODO 2: overwrite winner with (scope, m.group(1)) so the most specific one wins
                pass
    return winner

print("indentation for a project file:", indent_rule("talentdesk/screening/bar.py"))

**Self-check (offline).** The project (4) should override the user (2).

In [ ]:
# ===== self-check for TODO 2 =====
assert indent_rule("talentdesk/screening/bar.py") == ("project", "4"), "project 4-space overrides user 2-space"
assert indent_rule("talentdesk/util/text.py") == ("project", "4"), "same outside the screening folder"
print("TODO 2 checks passed")

**This cell:** a `/memory`-style view (provided). In the real CLI, `/memory` shows what is loaded
and lets you edit it. Here we list the active files for a target in load order, marking the directory
file as conditional so you can see why a rule did or did not apply.

In [ ]:
# ===== simulate the /memory listing (provided) =====
def memory_view(target):
    print(f"/memory for {target}:")
    for scope, src in effective_config(target):
        tag = " (on demand)" if scope.startswith("dir") else ""
        print(f"  [{scope}]{tag} {os.path.relpath(src, SANDBOX)}")
    print("  [rules] project/.claude/rules/*.md (path-scoped; see Part B)")

memory_view("talentdesk/screening/bar.py")
print()
memory_view("talentdesk/util/text.py")

---

### 🎯 Part B - load the right rules, and only those

Every `.md` in `.claude/rules/` loads as project memory. A rule **without** a `paths` field loads
**unconditionally**; a rule **with** a `paths` field (a YAML list of globs) loads **only** when Claude
works on a matching file. That conditional loading is where the context savings come from.

**TODO 3 (about 5 minutes).** Complete `glob_to_regex()`, the matcher. Translate each glob token to
regex: `**/` spans any folders, bare `**` spans anything, `*` stays within one path segment, `?` is one
non-slash char, and everything else is a literal (already handled).

In [ ]:
# ===== TODO 3 - match a path against a glob =====
def glob_to_regex(pat):                            # glob string -> compiled regex
    out = ""; i = 0
    while i < len(pat):
        if pat[i:i+3] == "**/":
            # 👉 TODO 3a: out += "(?:.*/)?"; i += 3      # ** plus slash spans directories
            i += 3
        elif pat[i:i+2] == "**":
            # 👉 TODO 3b: out += ".*"; i += 2            # bare ** spans anything
            i += 2
        elif pat[i] == "*":
            # 👉 TODO 3c: out += "[^/]*"; i += 1         # * stays within one segment
            i += 1
        elif pat[i] == "?":
            # 👉 TODO 3d: out += "[^/]"; i += 1          # ? is one non-slash char
            i += 1
        else:
            out += re.escape(pat[i]); i += 1            # literal char (provided)
    return re.compile("^" + out + "$")

def glob_match(pattern, path):                     # True if path matches the glob (provided)
    return glob_to_regex(pattern).match(path) is not None

print("bar.py   vs talentdesk/screening/**/*.py:", glob_match("talentdesk/screening/**/*.py", "talentdesk/screening/bar.py"))
print("test     vs **/test_*.py               :", glob_match("**/test_*.py", "tests/test_screening.py"))
print("text.py  vs talentdesk/screening/**/*.py:", glob_match("talentdesk/screening/**/*.py", "talentdesk/util/text.py"))

**Self-check (offline).**

In [ ]:
# ===== self-check for TODO 3 =====
assert glob_match("talentdesk/screening/**/*.py", "talentdesk/screening/bar.py") is True
assert glob_match("**/test_*.py", "tests/test_screening.py") is True
assert glob_match("talentdesk/screening/**/*.py", "talentdesk/util/text.py") is False
assert glob_match("talentdesk/scheduling/**/*.py", "talentdesk/scheduling/slots.py") is True
print("TODO 3 checks passed")

**This cell:** parse a rule into its **paths** and body (provided). The YAML frontmatter between
`---` fences gives the `paths` list; a rule with no frontmatter loads unconditionally.

In [ ]:
# ===== parse a rule into (paths, body) (provided) =====
def parse_rule(rel):                               # rule path -> (list_of_globs, body_text)
    text = open(os.path.join(PROJECT, rel)).read()
    m = re.match(r"^---\n(.*?)\n---\n(.*)$", text, re.DOTALL)   # fenced frontmatter?
    if not m:
        return [], text                             # no frontmatter -> unconditional
    front = yaml.safe_load(m.group(1)) or {}
    return front.get("paths", []), m.group(2)

for rel in [".claude/rules/screening.md", ".claude/rules/tests.md", ".claude/rules/style.md"]:
    paths, _ = parse_rule(rel)
    print(f"  {rel.split('/')[-1]:14} paths -> {paths or '(none: always loads)'}")

**TODO 4 (about 4 minutes).** Complete `rules_for()`, the resolver. A rule loads for a file if it
has **no** paths (unconditional) **or** if **any** of its globs match the file. Return the loaded rule
filenames in `RULES` order.

In [ ]:
# ===== TODO 4 - which rules load for a given file =====
RULES = [".claude/rules/screening.md", ".claude/rules/tests.md", ".claude/rules/style.md"]

def rules_for(file_path):                          # file -> the rule files that load for it
    loaded = []
    for rel in RULES:
        paths, _ = parse_rule(rel)
        # 👉 TODO 4: if paths is empty, OR any(glob_match(p, file_path) for p in paths),
        #            append rel.split("/")[-1] to loaded
        pass
    return loaded

for f in ["talentdesk/screening/bar.py", "tests/test_screening.py", "talentdesk/util/text.py"]:
    print(f"  {f:32} -> {rules_for(f)}")

**Self-check (offline).** Each file should load exactly its rules plus the unconditional style
rule.

In [ ]:
# ===== self-check for TODO 4 =====
EXPECTED = {
    "talentdesk/screening/bar.py": ["screening.md", "style.md"],
    "tests/test_screening.py":     ["tests.md", "style.md"],
    "talentdesk/util/text.py":     ["style.md"],
}
for f, want in EXPECTED.items():
    got = rules_for(f)
    assert got == want, f"{f}: expected {want}, got {got}"
print("TODO 4 checks passed")

**This cell:** the **payoff** (provided): modular versus monolithic. The monolithic file loads
every rule body for every file; the path-scoped setup loads only what applies. Tokens are approximated
as characters over four, totalled across the files.

In [ ]:
# ===== compare token cost: monolithic vs path-scoped (provided) =====
def body_tokens(rel):
    _, body = parse_rule(rel)
    return len(body) // 4

files = ["talentdesk/screening/bar.py", "tests/test_screening.py",
         "talentdesk/scheduling/slots.py", "talentdesk/util/text.py"]
monolithic_per_file = sum(body_tokens(r) for r in RULES)          # everything, every file
monolithic_total = monolithic_per_file * len(files)
modular_total = sum(sum(body_tokens(".claude/rules/" + name) for name in rules_for(f)) for f in files)

print("monolithic total tokens :", monolithic_total)
print("path-scoped total tokens:", modular_total)
print(f"path scoping saved {1 - modular_total / monolithic_total:.0%} across {len(files)} files")

**This cell:** the live view (provided). It points the Agent SDK at the project with `cwd=PROJECT`
and `setting_sources=["project"]`, which loads the project `CLAUDE.md` and its rules, then asks a
question the project memory answers. Offline it prints the expected outcome. The user layer is only
shown offline, so we never read your real home.

In [ ]:
# ===== live: load the project memory and ask =====
try:
    from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, TextBlock
    SDK_OK = True
    MEM_OPTS = ClaudeAgentOptions(model=MODEL, cwd=PROJECT,
                                  setting_sources=["project"],       # loads project CLAUDE.md + rules
                                  allowed_tools=["Read", "Grep", "Glob"])
    async def ask(prompt):
        async for m in query(prompt=prompt, options=MEM_OPTS):
            if isinstance(m, AssistantMessage):
                for b in m.content:
                    if isinstance(b, TextBlock) and b.text.strip(): print("  ", b.text.strip()[:160])
    print("project memory options ready (cwd =", PROJECT, ")")
except Exception:
    SDK_OK = False
    print("Agent SDK not available offline; the sandbox logic above was tested with Python.")

if RUN_LIVE and SDK_OK:
    run_async(lambda: ask("Based on this project's conventions, what indentation should I use? One line."))
    run_async(lambda: ask("Open talentdesk/screening/bar.py and summarize the screening rules that apply."))
else:
    print("[offline] expected: 4-space (project overrides user); and for bar.py the screening rule")
    print("          is in scope (ground decisions in candidate facts) plus the shared style rule.")

---

### Anti-patterns to avoid

| anti-pattern | what to do instead |
|---|---|
| put team conventions in `~/.claude/CLAUDE.md` | it is not shared; use the project file and rules |
| one giant `CLAUDE.md` | split with `@import` and move topics into `.claude/rules/` |
| assume a rule loaded | check with `/memory` |
| rely on `@import` to cut context | imports still load at launch; use path-scoped rules |
| many rule files with no `paths` | they all load at launch; add `paths` to scope them |
| path-scope rules in `~/.claude/rules/` | that is currently ignored; keep them project-level |

**Lesson:** project memory is layered: user, then project, then directory, with the most specific
winning. Because the user file is not shared, team conventions live in the project file and in
`.claude/rules/`. `@import` keeps files modular but does not save context; a **paths** field turns a
rule into a conditional one that loads only when Claude touches a matching file, keeping instructions
relevant and context small. And `/memory` is how you see what actually loaded.

---

## Recap - the configuration hierarchy and path scoping

| Layer / rule | File | Scope |
|---|---|---|
| User | `~/.claude/CLAUDE.md` | personal, not shared, loaded first |
| Project | `.claude/CLAUDE.md` | shared via Git, overrides user |
| Directory | subfolder `CLAUDE.md` | on demand, overrides project there |
| @import | `@standards/...` | organizes files; still loads at launch |
| screening.md | `paths: talentdesk/screening/**/*.py` | screening files only |
| tests.md | `paths: **/test_*.py` | test files only |
| style.md | (no paths) | every file |

**Try it next:** add a rule scoped to `talentdesk/scheduling/**/*.py` and confirm it loads for the
scheduling file but not the screening file. Then add a directory `CLAUDE.md` under `talentdesk/` and
watch which files it applies to.